# Cylinder stiffness ranking from microscope videos

Workflow: put all Fiji-processed videos in one folder, enter that folder in the setup cell, and use **Run All**. The folder may have any name. The default is a folder named `processed_videos` beside the notebook. Hough tracking supplies the cylinder search window; a globally continuous closed edge contour supplies its physical geometry. A second continuous edge path follows the pusher face, with an identity constraint preventing it from switching to the fixed wall.


## 1. One-time environment check

This cell contains and loads the complete analysis engine. It installs any missing third-party packages automatically; it does nothing when they are already installed. The temporary runtime files are generated automatically and do not need to be downloaded or managed.


In [9]:
from pathlib import Path
import base64
import hashlib
import importlib.util
import json
import subprocess
import sys
import tempfile

# Third-party libraries used by the embedded analyzer.
required = {
    "cv2": "opencv-python-headless>=4.9,<5",
    "numpy": "numpy>=1.26,<3",
    "pandas": "pandas>=2.0,<3",
    "scipy": "scipy>=1.11,<2",
    "matplotlib": "matplotlib>=3.8,<4",
    "openpyxl": "openpyxl>=3.1,<4",
    "tifffile": "tifffile>=2024.1",
    "nd2": "nd2>=0.10",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

# The complete local analysis package is embedded in this notebook.
_PAYLOAD_B64 = "..."
_EMBEDDED_FILES = json.loads(base64.b64decode(_PAYLOAD_B64).decode("utf-8"))
_RUNTIME_VERSION = "0446933eda6b"
RUNTIME_ROOT = Path(tempfile.gettempdir()) / f"cylinder_stiffness_notebook_{_RUNTIME_VERSION}"
for relative_name, contents in _EMBEDDED_FILES.items():
    destination = RUNTIME_ROOT / relative_name
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists() or destination.read_text(encoding="utf-8") != contents:
        destination.write_text(contents, encoding="utf-8")

# Always use this notebook's embedded version, even if an older project is installed.
for module_name in list(sys.modules):
    if module_name == "stiffness_analyzer" or module_name.startswith("stiffness_analyzer."):
        del sys.modules[module_name]
if str(RUNTIME_ROOT) in sys.path:
    sys.path.remove(str(RUNTIME_ROOT))
sys.path.insert(0, str(RUNTIME_ROOT))

print("Standalone analysis engine ready. No project download is required.")


Standalone analysis engine ready. No project download is required.


## 2. Experiment setup — edit only this cell

Set `INPUT_FOLDER` to the folder containing the Fiji-processed videos. 

Normally the automatic geometry is sufficient after a clean Fiji crop. If a preview is wrong, add an entry to `GEOMETRY_HINTS`. All geometry hints are fractions from 0 to 1 of the image width or height. Optional frame windows refer to original frame numbers.


In [10]:
# Folders — this is the only path you normally need to change.
# Default: a folder called "processed_videos" in Jupyter's current folder.
INPUT_FOLDER = Path("07 videos").expanduser().resolve()
RESULTS_FOLDER = Path("comparison_results_old notebook").expanduser().resolve()

# Experiment
PUSHER_SIDE = 'left'           # your movies: pusher-left / fixed-wall-right
THICKNESS_UM = 600.0
PIXEL_SIZE_UM = None           # e.g. 0.65; None keeps lengths in pixels
FRAME_INTERVAL_S = None        # e.g. 0.05; None uses movie metadata

# Analysis interval and edge sensitivity
MIN_STRAIN = 0.02
MAX_STRAIN = 0.35
FRAME_STRIDE = 1
THRESHOLD_SCALE = 0.75         # lower retains fainter edges
TRACKING_MODE = 'hough'        # Hough localization only; measurements use the image edges
MAX_AREA_CHANGE = 0.18

# Optional per-video geometry corrections. Uncomment and edit only if a preview is wrong.
GEOMETRY_HINTS = {
    # 'example.tif': {
    #     'cylinder_center_x_fraction': 0.50,
    #     'cylinder_center_y_fraction': 0.50,
    #     'cylinder_diameter_fraction': 0.25,
    #     'wall_x_fraction': 0.18,
    #     'pusher_x_fraction': 0.78,
    # },
}

# Optional reviewed loading windows: 'filename': (first_frame, last_frame)
FRAME_WINDOWS = {
    # 'example.tif': (12, 80),
}


## 3. Find videos and preview the detected geometry

Yellow: initial cylinder. Green: fixed wall. Cyan: far-field pusher face. If any preview is incorrect, edit `GEOMETRY_HINTS` above and rerun from this cell.

In [11]:
from dataclasses import replace
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from stiffness_analyzer.analysis import analyze_video, guess_geometry, rank_analyses
from stiffness_analyzer.io import discover_inputs, load_frames
from stiffness_analyzer.models import AnalysisConfig
from stiffness_analyzer.reporting import (
    build_results_bundle, curves_figure, excel_bytes, qc_figure, ranking_figure,
    strain_history_figure,
)

INPUT_FOLDER.mkdir(parents=True, exist_ok=True)
inputs = discover_inputs(INPUT_FOLDER)
if len(inputs) < 2:
    raise FileNotFoundError(
        f'Found {len(inputs)} supported file(s) in {INPUT_FOLDER}. Copy at least two processed videos there, then rerun this cell.'
    )
print(f'Found {len(inputs)} videos:')
for path in inputs:
    print('  -', path.name)

BASE_CONFIG = AnalysisConfig(
    pusher_side=PUSHER_SIDE, thickness_um=THICKNESS_UM,
    pixel_size_um=PIXEL_SIZE_UM, frame_interval_s=FRAME_INTERVAL_S,
    frame_stride=FRAME_STRIDE, threshold_scale=THRESHOLD_SCALE,
    tracking_mode=TRACKING_MODE,
    min_analysis_strain=MIN_STRAIN, max_analysis_strain=MAX_STRAIN,
    max_area_change_fraction=MAX_AREA_CHANGE,
)

VIDEO_CONFIGS = {}
preview_records = []
ncols = min(3, len(inputs))
nrows = math.ceil(len(inputs) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows), squeeze=False)

for ax, path in zip(axes.flat, inputs):
    supplied = GEOMETRY_HINTS.get(path.name, {})
    preview_config = replace(BASE_CONFIG, max_frames=5, **supplied)
    frames, metadata = load_frames(path, preview_config)
    auto = guess_geometry(frames, preview_config)
    automatic = auto.normalized(metadata.width_px, metadata.height_px)
    geometry = {**automatic, **supplied}
    start_end = FRAME_WINDOWS.get(path.name, (None, None))
    config = replace(
        BASE_CONFIG, **geometry, start_frame=start_end[0], end_frame=start_end[1]
    )
    VIDEO_CONFIGS[path.name] = config

    frame = frames[0]
    height, width = frame.shape[:2]
    cx = config.cylinder_center_x_fraction * width
    cy = config.cylinder_center_y_fraction * height
    diameter = config.cylinder_diameter_fraction * width
    ax.imshow(frame, cmap='gray')
    ax.add_patch(plt.Circle((cx, cy), diameter / 2, fill=False, color='#FFD200', lw=2))
    ax.axvline(config.wall_x_fraction * width, color='#2BD12B', lw=2)
    ax.axvline(config.pusher_x_fraction * width, color='#00D5FF', lw=2)
    ax.set_title(path.name)
    ax.axis('off')
    preview_records.append({
        'video': path.name, 'automatic_confidence': auto.confidence,
        'diameter_px_guess': diameter, 'notes': ' | '.join(auto.notes)
    })
for ax in axes.flat[len(inputs):]:
    ax.axis('off')
plt.tight_layout()
plt.show()
display(pd.DataFrame(preview_records))

Found 3 videos:
  - pushing07_07_60.tif
  - pushingdefault.tif
  - v2_block=cylinder.tif


C:\Users\user\AppData\Local\Temp\ipykernel_18032\213028195.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,video,automatic_confidence,diameter_px_guess,notes
0,pushing07_07_60.tif,0.941526,276.239990,
1,pushingdefault.tif,0.932245,239.520020,
2,v2_block=cylinder.tif,0.981814,279.119995,


## 4. Analyze every video

The code detects first simultaneous wall–cylinder–pusher contact, traces the closed cylinder edge without imposing an ellipse, measures its direct horizontal span, excludes rigid translation and unloading, and applies area-conservation and edge-support checks.

In [12]:
analyses = []
failures = []
for number, path in enumerate(inputs, start=1):
    print(f'[{number}/{len(inputs)}] Analyzing {path.name} ...', flush=True)
    try:
        analysis = analyze_video(path, VIDEO_CONFIGS[path.name], keep_overlay_frames=True)
        analyses.append(analysis)
    except Exception as error:
        failures.append({'video': path.name, 'error': str(error)})

if failures:
    print('Failures:')
    display(pd.DataFrame(failures))
if len(analyses) < 2:
    raise RuntimeError('Fewer than two videos were analyzed successfully.')
summary = pd.DataFrame([analysis.summary_dict() for analysis in analyses])
display(summary)

[1/3] Analyzing pushing07_07_60.tif ...
[2/3] Analyzing pushingdefault.tif ...
[3/3] Analyzing v2_block=cylinder.tif ...


,video,initial_diameter_px,initial_diameter_um,wall_x_px,contact_start_frame,valid_start_frame,valid_end_frame,valid_frames,quality_score,median_cylinder_confidence,hough_tracked_fraction,tracking_fallback_fraction,median_area_ratio,wall_clipped_frame_fraction,max_ellipse_wall_overshoot_px,median_edge_support_fraction,median_strain_rate_s_1,warnings
0,pushing07_07_60.tif,260.578361,None,295.5,1,1,43,18,0.581289,0.9900,1.0,0.0,0.984496,1.0,31.605757,0.986111,None,"The movie starts at two-sided contact, so the ..."
1,pushingdefault.tif,245.734709,None,269.0,0,6,18,13,0.714668,0.9825,1.0,0.0,1.049847,1.0,57.860850,0.975694,None,"The movie starts at two-sided contact, so the ..."
2,v2_block=cylinder.tif,270.591394,None,302.0,2,2,49,33,0.625768,0.9900,1.0,0.0,0.931619,1.0,36.537279,0.986111,None,The pusher interface was detected with low con...


## 5. Plot horizontal strain for all videos

This plot places every video's direct horizontal strain, $\varepsilon_x=(D_0-D_x)/D_0$, on the same axes. Faint lines show the complete tracked trajectory; bold lines show the valid loading frames used for comparison.

In [13]:
display(strain_history_figure(analyses))

<Figure size 1000x600 with 1 Axes>

## 6. Review representative diagnostic overlays

Yellow is the direct cylinder edge contour, green the fixed-wall line, cyan the detected pusher surface, and magenta its inferred unindented reference. The ellipse localization guide is intentionally not drawn and is not used for strain. The overlay prints direct horizontal cylinder strain and edge support.

In [14]:
fig, axes = plt.subplots(len(analyses), 2, figsize=(12, 4.5 * len(analyses)), squeeze=False)
for row, analysis in enumerate(analyses):
    valid_indices = analysis.measurements.index[analysis.measurements['valid_for_ranking']].tolist()
    selected = [valid_indices[0], valid_indices[-1]] if valid_indices else [0, len(analysis.overlay_frames) - 1]
    for column, index in enumerate(selected):
        axes[row, column].imshow(analysis.overlay_frames[index][..., ::-1])
        source_frame = int(analysis.measurements.iloc[index]['source_frame'])
        axes[row, column].set_title(f'{analysis.name} — frame {source_frame}')
        axes[row, column].axis('off')
plt.tight_layout()
plt.show()
for analysis in analyses:
    for warning in analysis.warnings:
        print(f'WARNING — {analysis.name}: {warning}')

WARNING — pushing07_07_60.tif: The movie starts at two-sided contact, so the undeformed diameter uses the plane-strain area estimate. Include several pre-contact frames if possible.
WARNING — pushing07_07_60.tif: The pusher interface was detected with low confidence.
WARNING — pushingdefault.tif: The movie starts at two-sided contact, so the undeformed diameter uses the plane-strain area estimate. Include several pre-contact frames if possible.
WARNING — pushingdefault.tif: The cylinder translates vertically by more than the configured diagnostic limit. The outline remains usable, but compare contact geometry across videos.
WARNING — v2_block=cylinder.tif: The pusher interface was detected with low confidence.


C:\Users\user\AppData\Local\Temp\ipykernel_18032\696025438.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Calculate and inspect the ranking

The ranking uses normalized pusher-indentation area as a common load coordinate. At matched pusher deformation, less direct horizontal cylinder strain means a stiffer cylinder.

In [15]:
ranking_result = rank_analyses(analyses)
print('STRAIN RANKING — smaller strain at matched pusher deformation is stiffer')
display(ranking_result.strain_ranking.round(4))
if ranking_result.common_load_min is not None:
    print(f'Common normalized pusher-deformation interval: {ranking_result.common_load_min:.4f} to {ranking_result.common_load_max:.4f}')
for warning in ranking_result.warnings:
    print('WARNING:', warning)
display(ranking_figure(ranking_result))
display(curves_figure(ranking_result))
display(qc_figure(analyses))

STRAIN RANKING — smaller strain at matched pusher deformation is stiffer


,strain_rank,video,quality_score,valid_frames,mean_cylinder_strain_at_common_load,strain_stiffness_score,strain_index_softest_1
0,1,pushing07_07_60.tif,0.5813,18,0.1865,5.3625,1.5245
1,2,v2_block=cylinder.tif,0.6258,33,0.2591,3.8598,1.0973
2,3,pushingdefault.tif,0.7147,13,0.2843,3.5176,1.0000


Common normalized pusher-deformation interval: 0.0001 to 0.2437


<Figure size 900x450 with 1 Axes>

<Figure size 900x550 with 1 Axes>

<Figure size 1200x800 with 4 Axes>

## 8. Save the complete report

The ZIP contains the strain ranking, common-load strain curves, the combined all-video strain plot, frame-level edge measurements, quality-control figures, settings, warnings, and one complete diagnostic overlay movie per input.

In [16]:
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)
bundle_path = RESULTS_FOLDER / 'stiffness_results.zip'
excel_path = RESULTS_FOLDER / 'stiffness_results.xlsx'
csv_path = RESULTS_FOLDER / 'strain_ranking.csv'
strain_plot_path = RESULTS_FOLDER / 'strain_all_videos.png'
bundle_path.write_bytes(build_results_bundle(analyses, ranking_result, include_overlay_videos=True))
excel_path.write_bytes(excel_bytes(analyses, ranking_result))
ranking_result.strain_ranking.to_csv(csv_path, index=False)
strain_export_figure = strain_history_figure(analyses)
strain_export_figure.savefig(strain_plot_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(strain_export_figure)
print('Saved:')
print(' -', bundle_path.resolve())
print(' -', excel_path.resolve())
print(' -', csv_path.resolve())
print(' -', strain_plot_path.resolve())

Saved:
 - C:\Users\user\LadHyx_project\comparison_results_old notebook\stiffness_results.zip
 - C:\Users\user\LadHyx_project\comparison_results_old notebook\stiffness_results.xlsx
 - C:\Users\user\LadHyx_project\comparison_results_old notebook\strain_ranking.csv
 - C:\Users\user\LadHyx_project\comparison_results_old notebook\strain_all_videos.png


##### Interpretation checklist

Accept an ordering only when the yellow contour follows the actual cylinder edge, each video contributes enough valid frames, and loading rates are comparable. If the strain curves cross, report the ranking as strain-dependent rather than forcing one unconditional order.